In [ ]:
import random
import numpy as np
import h5py

# 定义模块的公开接口
__all__ = ["SignalDataLoader"]


class SignalDataLoader(object):
    '''
    从 HDF5 文件加载通信信号数据集，并按照信噪比 (SNR) 和调制类型进行分割，用于深度学习模型的训练和评估。
    '''
    def __init__(self, mod_type=[]):
        # 定义所有可能的调制类型（共25种）
        # 包括常见的数字调制和模拟调制方式
        mods = ["BPSK", "QPSK", "8PSK", "PAM4", "QAM16", "QAM32", "QAM64", "QAM128", "QAM256", "GFSK", "WBFM", "AM-DSB",
                "AM-SSB", "OOK", "4ASK", "8ASK", "16PSK", "32PSK", "8APSK", "GMSK", "DQPSK", "16APSK", "32APSK",
                "64APSK", "128APSK"]  # "CPFSK"

        # 从HDF5文件加载数据
        # X: 信号数据 (IQ样本)
        # cfo:
        # mod: 调制类型索引
        # snr: 信噪比
        # sps:
        # sro:
        # 'r+'：r：只读模式、+：允许后续写入（虽然此处未使用写入功能）
        with h5py.File('dataset/cfo.hdf5', 'r+') as h5file:

            # h5file['X']：访问HDF5文件中名为 X 的 dataset（数据集）
            # [:] 操作：将整个数据集加载到内存，返回一个NumPy数组
            # np.asarray 转换：将HDF5数据集对象显式转换为NumPy数组
            #   HDF5数据集对象是“虚拟”的，不直接支持所有NumPy操作
            #   转换后可以释放HDF5文件句柄，减少资源占用
            # allX：(500000, 2, 1024)
            allX = np.asarray(h5file['X'][:])

            # h5file['mod'][:]：读取调制类型索引（通常是整数，如 0 代表BPSK，1 代表QPSK等）
            # [mods[i] for i in h5file['mod'][:]]：将整数索引映射到具体的调制名称（如 0 → "BPSK"）
            # allY：(500000, )
            allY = np.asarray([mods[i] for i in h5file['mod'][:]])


            # 直接读取SNR值，通常为浮点数组（单位：dB）
            # allZ：(500000, )
            allZ = np.asarray(h5file['snr'][:])

        # 根据用户指定的调制类型筛选数据
        X = []
        Y = []
        Z = []

        for idx in range(allX.shape[0]):
            if allY[idx] in mod_type:
                X.append(allX[idx])
                # 将调制类型转换为索引(从0开始)
                Y.append(mod_type.index(allY[idx]))
                Z.append(allZ[idx])
        # 删除原始数据以释放内存
        del allX
        del allY
        del allZ
        X = np.asarray(X)
        Y = np.asarray(Y)
        Z = np.asarray(Z)

        # 提取唯一的信噪比列表和调制类型列表
        self.snrs = np.unique(Z).tolist()
        self.mods = mod_type

        # 数据集分割
        # 50% 训练集, 25% 验证集, 25% 测试集
        n_examples = X.shape[0]
        n_train = int(0.5 * n_examples)
        n_valid = int(0.25 * n_examples)

        # 随机打乱数据索引
        allnum = list(range(0, n_examples))
        random.shuffle(allnum)

        # 分割索引
        train_idx = allnum[0:n_train]
        valid_idx = allnum[n_train:n_train + n_valid]
        test_idx = allnum[n_train + n_valid:]

        # 根据索引提取数据
        self.X_train = X[train_idx]
        self.Y_train = Y[train_idx]
        self.Z_train = Z[train_idx]
        self.X_valid = X[valid_idx]
        self.Y_valid = Y[valid_idx]
        self.Z_valid = Z[valid_idx]
        self.X_test = X[test_idx]
        self.Y_test = Y[test_idx]
        self.Z_test = Z[test_idx]
        del X
        del Y
        del Z

    def __call__(self):
        # 实现可调用对象协议
        # 返回所有加载和分割的数据
        return self.X_train, self.Y_train, self.Z_train, self.X_valid, self.Y_valid, self.Z_valid, self.X_test, self.Y_test, self.Z_test, self.snrs, self.mods